# DS403 · Homework 1 — data manipulation on sessions 1–4

> **Work in your own copy** (from day 1): *File → Save a copy in Drive*, then *File → Move* it into **DS403 students → your folder** (your first and last name). Work only there: your teachers follow your progress from that folder. In the exercise cells, **stop a little and think** before you type, and solve them **without consulting generative systems**.


**Assigned at session 5 · Due at the start of session 9, Thursday 29 October (09:00) · weight 20% · AI-free.** The syllabus asks for no
generative help on this homework: it is practice for the unaided exam. Turn completion off for this notebook; if
you used any help, say so in the cell below. There is no penalty for saying so.

**Hand-in:** push this notebook to your repository and tag the commit `hw1` before the deadline. It must pass
*Restart & Run All*.

Each question names what to compute; put your final answer in its `answer_…` variable, and keep the code that
computes it.

**How it is built.** It follows what we actually did in class, so that you can do every graded task on your own:

* **Part A · What we practised in class**: Sessions 2–4 as the room did them. Each task names the block that taught it: reopen that block first if you are stuck.
* **Part B · New steps, with a worked example**: Session 3's second half (merge, gaps, sessions), which we did not reach in class. Each task starts with a small worked example on six rows: run it, change it, then do the same on the real data.
* **Part C · Stretch (optional)** (not graded): Session 4's last blocks. Not needed for full marks; do them if A and B are done, they come back in the exam.

**When you are stuck:** open the hint under the task, then the block it names in that session's notebook (yours, in
your folder). The *checkpoint* cells check a step on the way, not the answer: `[x]` means you are on track. Then ask
the course library, or bring the question to class. After you hand in, you explain a few of your own cells in the
whiteboard interview, so write the code you can explain.

In [1]:
# Disclosure: did you use any AI or other help? Say what, and for which question.
DISCLOSURE = ""

In [2]:
#@title Course setup — run this first (on Colab it prepares Drive, pandas and the data; on a laptop it does nothing)
import os, shutil, subprocess, sys
from pathlib import Path

COURSE_FOLDER = "261019 HSIT Generative Python product development"   # the shared course folder on Google Drive
COURSE_FOLDER_ID = "1z3bZ5zoE6mE7-oyLk1gmBDXGrEVFTlXd"

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    from importlib.metadata import version
    if int(version("pandas").split(".")[0]) < 3:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas>=3.0,<3.1", "pyarrow"], check=True)
        if "pandas" in sys.modules:   # Colab loaded the old pandas already: only a restart replaces it
            raise SystemExit("pandas 3 is installed. Now: Runtime → Restart session, then run this cell again.")
    import pandas as pd
    home = Path("/content/drive/MyDrive/DS403 students")
    data = home / "data"
    if not (data / "events.csv").exists():
        drive_root = Path("/content/drive")
        # By the folder's id, never by searching a whole Drive (slow, and it depends on where you keep things):
        # the shortcut in My Drive (wherever you put it, Drive lists its target by id), or the folder itself in My Drive.
        src = next((c for c in (drive_root / ".shortcut-targets-by-id" / COURSE_FOLDER_ID / COURSE_FOLDER / "data",
                                drive_root / "MyDrive" / COURSE_FOLDER / "data") if (c / "events.csv").exists()), None)
        data.mkdir(parents=True, exist_ok=True)
        if src:
            for f in ("events.csv", "audience.csv"):
                shutil.copy(src / f, data / f)
        else:
            # No shortcut: read the two files straight from the course folder (Google asks once for permission).
            from google.colab import auth
            from googleapiclient.discovery import build
            from googleapiclient.http import MediaIoBaseDownload
            auth.authenticate_user()
            api = build("drive", "v3")
            def children(parent, extra=""):
                return api.files().list(q=f"'{parent}' in parents and trashed = false{extra}", fields="files(id,name)",
                                        supportsAllDrives=True, includeItemsFromAllDrives=True).execute()["files"]
            files = []
            try:   # the folder named 'data' that holds the files (Drive allows two folders with one name)
                for d in children(COURSE_FOLDER_ID, " and name = 'data'"):
                    files = children(d["id"])
                    if any(f["name"] == "events.csv" for f in files):
                        break
                    files = []
            except Exception:
                files = []
            if not files:
                raise FileNotFoundError(f"You cannot open the course folder '{COURSE_FOLDER}' yet: ask your teacher to "
                                        "share it with you, then run this cell again.")
            for f in files:
                if f["name"] in ("events.csv", "audience.csv"):
                    with open(data / f["name"], "wb") as out:
                        down = MediaIoBaseDownload(out, api.files().get_media(fileId=f["id"]))
                        done = False
                        while not done:
                            _, done = down.next_chunk()
        print("The course data is copied to", data)
    # Work from a local folder beside a link to your data on Drive: '../data' then means your ds403/data, exactly as
    # on a laptop (inside the Drive mount itself, '..' does not resolve), and what a notebook saves still lands on Drive.
    local = Path("/content/ds403")
    (local / "work").mkdir(parents=True, exist_ok=True)
    if not (local / "data").exists():
        (local / "data").symlink_to(data, target_is_directory=True)
    os.chdir(local / "work")
    print("Ready on Colab · pandas", pd.__version__, "· data in", data)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Ready on Colab · pandas 3.0.6 · data in /content/drive/MyDrive/DS403 students/data


In [3]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA = Path("../data")
raw_events = pd.read_csv(DATA / "events.csv")      # timestamps as text: parse them yourself (q01)
audience = pd.read_csv(DATA / "audience.csv")
raw_events.shape, audience.shape

((334438, 6), (5000, 7))

In [4]:
def checkpoint(label, compute, expected):
    """A check of a step on the way, never of the answer: [x] means you are on track. It never raises."""
    try:
        got = compute()
    except NameError as err:
        print(f"[ ] {label}: not yet ({err.args[0]})")
        return
    except Exception as err:                     # still None, or half built: say so, keep the notebook running
        print(f"[ ] {label}: not yet ({'fill in the variable above first' if 'NoneType' in str(err) else f'{type(err).__name__}: {err}'})")
        return
    print(f"[x] {label}" if got == expected else f"[ ] {label}: not yet (you have {got!r})")

# Part A · What we practised in class

Sessions 2–4 as the room did them. Each task names the block that taught it: reopen that block first if you are stuck.

## q01 · Parse `ts` of the raw events as ISO 8601 only (`errors="coerce"`). How many rows are left as NaT? Then parse both formats into `events_t`, one timestamp per row.

*Taught in:* session 2, block 10 (Reading the two CSVs — and parsing time).

<details><summary>Hint (open it only after a first try)</summary>

`pd.to_datetime(…, format="ISO8601", errors="coerce", utc=True)`, then `.isna().sum()`. The rows left are the older day/month/year export: parse them with their own `format=` and `fillna` the first result with it.

</details>

In [5]:
# your code
iso = pd.to_datetime(raw_events["ts"],
                     format = "ISO8601",
                     errors = "coerce",
                     utc = True)

old = pd.to_datetime(raw_events['ts'],
                     format='%d/%m/%Y %H:%M:%S',
                     errors="coerce",
                     utc=True)

events_t = iso.fillna(old)      # keep this name: the checkpoint below reads it
answer_q01 = int(iso.isna().sum())

In [6]:
checkpoint('after parsing both formats, no timestamp is missing', lambda: int(events_t.isna().sum()), 0)

[x] after parsing both formats, no timestamp is missing


## q02 · On the raw events table: how many rows does `.loc[10:20]` return, and how many does `.iloc[10:20]`? Answer as two numbers, label first.

*Taught in:* session 2, block 12 (.loc by label, .iloc by position).

<details><summary>Hint (open it only after a first try)</summary>

One includes its stop, the other does not. Check with `len(...)` rather than by eye.

</details>

In [7]:
# your code
answer_q02 = len(raw_events.loc[10:20]), len(raw_events.iloc[10:20])

## q03 · The audience table has one row per user. How many distinct users appear in the events table?

*Taught in:* session 2, block 14 (Two tables: an event, a user).

<details><summary>Hint (open it only after a first try)</summary>

`nunique()` on the column that names the user.

</details>

In [8]:
# my code
# print(len(raw_events["user_id"]))
# print(len(raw_events["user_id"].unique()))
# print(len(set(raw_events["user_id"])))

# print(audience["user_id"]).nunique()

In [9]:
# your code

answer_q03 = raw_events["user_id"].nunique()

In [10]:
checkpoint('the audience table: at most one row per user', lambda: int(audience['user_id'].value_counts().max()), 1)

[x] the audience table: at most one row per user


## q04 · Clean first: put your timestamps in as a column `t` (`raw_events["t"] = events_t`). How many exact duplicate rows are there? Drop them and keep the result as `events`: every task below uses it.

*Taught in:* session 3, block 1 (Clean first, then ask: is this tidy?).

<details><summary>Hint (open it only after a first try)</summary>

`duplicated()` marks a row identical in every column to an earlier one; `drop_duplicates(ignore_index=True)` keeps the first copy.

</details>

In [11]:
# your code

raw_events["t"] = events_t
events = raw_events.drop_duplicates().reset_index(drop = True)

answer_q04 = int(raw_events.duplicated().sum())

## q05 · Events per user. Users with more than 1,000 events are bots. How many bots are there, and which human user (1,000 events or fewer) is the most active? Answer as two values, e.g. `12, u00001`.

*Taught in:* session 3, block 7 (Split, apply, combine); in class: bots out before idxmax.

<details><summary>Hint (open it only after a first try)</summary>

`groupby('user_id').size()` gives a Series indexed by user; a boolean mask on it splits bots from humans; `idxmax()` names the top one.

</details>

In [12]:
# your code

per_user = events.groupby("user_id").size()    # keep this name: the checkpoint below reads it
answer_q05 = int((per_user > 1000).sum()), per_user[per_user <=1000].idxmax()

In [13]:
checkpoint('per-user counts add up to every event after cleaning', lambda: int(per_user.sum()), 332902)

[x] per-user counts add up to every event after cleaning


## q06 · Count events per day and event type as a wide table (days down, the 6 event types across), then melt it back to long. How many rows does the long table have?

*Taught in:* session 3, blocks 2–3 (pivot vs pivot_table, melt).

<details><summary>Hint (open it only after a first try)</summary>

A day is `t.dt.floor("D")`. `pivot_table(index=…, columns=…, values=…, aggfunc="count")`, then `reset_index().melt(id_vars=…)`. Long rows = days × event types.

</details>

In [14]:
# your code

events["day"] = events["t"].dt.floor("D")

wide = events.pivot_table(
    index = "day",
    columns = "event",
    values = "user_id",
    aggfunc = "count"
)      # keep this name: the checkpoint below reads it

long_table = wide.reset_index().melt(id_vars = "day")

answer_q06 = len(long_table)

In [15]:
checkpoint('the wide table has one column per event type', lambda: wide.shape[1], 6)

[x] the wide table has one column per event type


## q07 · Give every event row its user's total number of events (`transform`). What share of all events, in % with one decimal, comes from the bots (users with more than 1,000 events)?

*Taught in:* session 3, block 8 (Group statistics back on every row: transform).

<details><summary>Hint (open it only after a first try)</summary>

`transform('size')` returns one value per row, aligned with the original index; then the mean of a boolean column is a share.

</details>

In [16]:
# your code

events_n = events.groupby("user_id")["event"].transform("size")      # keep this name: the checkpoint below reads it
share = float(((events_n > 1000).sum() / len(events_n)) * 100)

answer_q07 = round(share, 1)

In [17]:
checkpoint('transform keeps one row per event (agg would not)', lambda: len(events_n) == len(events), True)

[x] transform keeps one row per event (agg would not)


## q08 · With numpy: build a users × weeks boolean matrix, True where the user had at least one event that week (week 0 starts on 1 June 2026, 00:00 UTC). Which week index has the most active users?

*Taught in:* session 4, block 4 (Hands-on: who is active, week by week).

<details><summary>Hint (open it only after a first try)</summary>

Week = `(t - start).dt.days // 7`; users to row numbers with `pd.factorize`; set `active[rows, weeks] = True`; then sum along the axis that runs over users.

</details>

In [18]:
# your code
start = pd.Timestamp("2026-06-01", tz='UTC')
weeks = (events["t"] - start).dt.days // 7

rows, users = pd.factorize(events["user_id"])

active = np.zeros(
    (len(users), weeks.max() + 1),
    dtype = bool
)      # keep this name: the checkpoint below reads it
active[rows, weeks] = True
answer_q08 = int(active.sum(axis = 0).argmax())

In [19]:
checkpoint('4,299 users with events × 17 weeks', lambda: active.shape, (4299, 17))

[x] 4,299 users with events × 17 weeks


## q09 · Make the same matrix with event counts instead of True/False, and divide each row by that user's mean events per week. What is the shape of the mean you divide by, so that broadcasting works per user?

*Taught in:* session 4, blocks 10–11 (Vectorisation; the broadcasting rule); in class: keepdims.

<details><summary>Hint (open it only after a first try)</summary>

`np.add.at(counts, (rows, weeks), 1)` counts repeated pairs (plain `+=` would count each pair once). Then print `counts.mean(axis=1).shape` with and without `keepdims=True`: only one of the two divides each row by its own mean.

</details>

In [20]:
# your code
n_users = len(users)
n_weeks = weeks.max() + 1
counts = np.zeros((n_users, n_weeks), dtype = int)
np.add.at(counts, (rows, weeks), 1)

answer_q09 = counts.mean(axis = 1, keepdims = True).shape

## q10 · With numpy only: take the hour (UTC) of every human user's event and count events per hour with `np.bincount`. Which hour of the day is the busiest?

*Taught in:* session 4, blocks 10 and 13 (vectorised hour, bincount; boolean masks: bots and night owls).

<details><summary>Hint (open it only after a first try)</summary>

Hours as an array: `events['t'].dt.hour.to_numpy()`; a boolean array of 'is human' selects the rows; `bincount(…, minlength=24).argmax()`.

</details>

In [21]:
# your code
hours = events["t"].dt.hour.to_numpy()
human_hours = hours[events_n <= 1000]
hour_counts = np.bincount(human_hours, minlength = 24)
answer_q10 = hour_counts.argmax()

# Part B · New steps, with a worked example

Session 3's second half (merge, gaps, sessions), which we did not reach in class. Each task starts with a small worked example on six rows: run it, change it, then do the same on the real data.

## q11 · Put each user's event count onto the audience table with `merge(how="left", validate=…)`. Which `validate` value states one row per user on both sides, and how many users have no events (NaN count)? Answer e.g. `one_to_one, 12`.

*Taught in:* session 3, blocks 10–11 (merge and how=; validate= catches row explosions) — not reached in class.

<details><summary>Hint (open it only after a first try)</summary>

The audience is the left table: every user stays. Users without events get NaN; count them with `.isna().sum()`.

</details>

**Worked example.** Six events of two users, and an audience of three: `c` signed up and never came back. `how="left"` keeps every user of the left table; `validate` makes pandas refuse a merge that would multiply rows.

In [22]:
toy = pd.DataFrame({"user_id": ["a", "b", "a", "a", "b", "a"],
                    "t": pd.to_datetime(["2026-06-01 10:00", "2026-06-01 09:00", "2026-06-01 10:05",
                                         "2026-06-01 11:00", "2026-06-01 09:02", "2026-06-01 10:01"], utc=True)})
users_toy = pd.DataFrame({"user_id": ["a", "b", "c"], "channel": ["organic", "referral", "organic"]})

counts_toy = toy.groupby("user_id").size().rename("n_events").reset_index()      # a: 4, b: 2
merged_toy = users_toy.merge(counts_toy, on="user_id", how="left", validate="one_to_one")
display(merged_toy)                          # c is still there, with NaN: no events

# Now break it on purpose: a user twice on the right would silently add a row; validate refuses.
try:
    users_toy.merge(pd.concat([counts_toy, counts_toy.head(1)]), on="user_id", how="left", validate="one_to_one")
except pd.errors.MergeError as err:
    print("validate caught it:", err)

,user_id,channel,n_events
0,a,organic,4.0
1,b,referral,2.0
2,c,organic,NaN


validate caught it: Merge keys are not unique in right dataset; not a one-to-one merge
Duplicates in right:
 user_id
      a ...


In [23]:
# your code

users = audience.merge(per_user.rename("n_events").reset_index(),
                       on = "user_id", how = "left",
                       validate="one_to_one")      # keep this name: the checkpoint below reads it
answer_q11 = ("one_to_one", int(users["n_events"].isna().sum()))

In [24]:
checkpoint('the merge kept exactly one row per user of the audience', lambda: len(users), 5000)

[x] the merge kept exactly one row per user of the audience


## q12 · Sort the events by user, then time. Per user, the gap between each event and the one before it, in minutes: what is the median gap (one decimal)?

*Taught in:* session 3, block 12 (A log in time: sort, then diff) — not reached in class.

<details><summary>Hint (open it only after a first try)</summary>

`sort_values(['user_id', 't'])`, then `groupby('user_id')['t'].diff()`: each user's first event has no gap (NaT). `.dt.total_seconds() / 60` gives minutes.

</details>

**Worked example.** The same six events. Sort first, by user and then by time: `diff` subtracts the row above, so an unsorted log gives nonsense gaps, and without `groupby` the first event of `b` would be measured from the last event of `a`.

In [25]:
toy = toy.sort_values(["user_id", "t"])
toy["gap_min"] = toy.groupby("user_id")["t"].diff().dt.total_seconds() / 60
toy                                          # a: NaN, 1, 4, 55 · b: NaN, 2

,user_id,t,gap_min
0,a,2026-06-01 10:00:00+00:00,NaN
5,a,2026-06-01 10:01:00+00:00,1.0
2,a,2026-06-01 10:05:00+00:00,4.0
3,a,2026-06-01 11:00:00+00:00,55.0
1,b,2026-06-01 09:00:00+00:00,NaN
4,b,2026-06-01 09:02:00+00:00,2.0


In [26]:
# your code
events = events.sort_values(["user_id", "t"], ignore_index=True)
gaps = events.groupby("user_id")["t"].diff().dt.total_seconds() / 60      # keep this name: the checkpoint below reads it
answer_q12 = round(gaps.median(), 1)

In [27]:
checkpoint('one missing gap per user: their first event', lambda: int(gaps.isna().sum()), 4299)

[x] one missing gap per user: their first event


## q13 · A new session starts at a user's first event or after more than 30 minutes without events. How many sessions are there in total?

*Taught in:* session 3, block 13 (Hands-on: sessions from gaps) — not reached in class.

<details><summary>Hint (open it only after a first try)</summary>

`new = gaps.isna() | (gaps > 30)`; a running count of `new` per user numbers the sessions: `new.groupby(user).cumsum()`. A user's sessions = their largest number.

</details>

**Worked example.** Mark where a session starts (the first event, or a gap over 30 minutes), then count the starts as you go down each user's events: the running count is the session's number.

In [28]:
toy["new_session"] = toy["gap_min"].isna() | (toy["gap_min"] > 30)
toy["session_no"] = toy.groupby("user_id")["new_session"].cumsum()
display(toy)                                 # a: 1, 1, 1, 2 · b: 1, 1
toy.groupby("user_id")["session_no"].max().sum()   # 2 sessions for a, 1 for b: 3

,user_id,t,gap_min,new_session,session_no
0,a,2026-06-01 10:00:00+00:00,NaN,True,1
5,a,2026-06-01 10:01:00+00:00,1.0,False,1
2,a,2026-06-01 10:05:00+00:00,4.0,False,1
3,a,2026-06-01 11:00:00+00:00,55.0,True,2
1,b,2026-06-01 09:00:00+00:00,NaN,True,1
4,b,2026-06-01 09:02:00+00:00,2.0,False,1


np.int64(3)

In [29]:
# your code

new_session = gaps.isna() | (gaps > 30)      # keep this name: the checkpoint below reads it
session_no = new_session.groupby(events["user_id"]).cumsum()
answer_q13 = int(new_session.sum())

In [30]:
checkpoint('at least one session start per user', lambda: int(new_session.sum()) >= 4299, True)

[x] at least one session start per user


## q14 · Sessions per user onto the audience, users with no events counted as 0. What is the mean number of sessions per user for acquisition_channel = 'paid_social' (two decimals)?

*Taught in:* session 3, block 15 (Hands-on: per-user features onto the audience) — not reached in class.

<details><summary>Hint (open it only after a first try)</summary>

Your merge from q11 again, with sessions instead of events; `fillna(0)` before the mean, or the users without events silently drop out.

</details>

**Worked example.** Sessions per user onto the audience. A user with no events has no row in the events, so after the left merge they are NaN: decide that NaN means 0 sessions before you average.

In [31]:
sessions_toy = toy.groupby("user_id")["session_no"].max().rename("sessions").reset_index()
per_user_toy = users_toy.merge(sessions_toy, on="user_id", how="left", validate="one_to_one")
print(per_user_toy.groupby("channel")["sessions"].mean())             # organic 2.0: c silently left out
per_user_toy["sessions"] = per_user_toy["sessions"].fillna(0)
print(per_user_toy.groupby("channel")["sessions"].mean())             # organic 1.0: c counted as 0

channel
organic     2.0
referral    1.0
Name: sessions, dtype: float64
channel
organic     1.0
referral    1.0
Name: sessions, dtype: float64


In [32]:
# your code
sessions = session_no.groupby(events["user_id"]).max().rename("sessions").reset_index()
users = users.merge(sessions,
                    on = "user_id",
                    how = "left",
                    validate = "one_to_one")
users["sessions"] = users["sessions"].fillna(0)
answer_q14 = round(users.loc[users["acquisition_channel"] == "paid_social", "sessions"].mean(), 2)

# Part C · Stretch (optional)

Session 4's last blocks. Not needed for full marks; do them if A and B are done, they come back in the exam.

## q15 · Optional. With numpy only (your gaps from q12 as an array, NaN removed): how many gaps are longer than 60 seconds?

*Taught in:* session 4, blocks 14–15 (cumsum and diff; inter-event times for every user).

<details><summary>Hint (open it only after a first try)</summary>

`g = gaps.to_numpy(); g = g[~np.isnan(g)]`; the gaps are in minutes, so 60 seconds is 1 minute; the sum of a boolean array counts its True.

</details>

In [33]:
# your code
g = gaps.to_numpy()
g = g[~np.isnan(g)]
answer_q15 = int((g>1).sum())

## q16 · Optional. Bin the same gaps into [0, 1), [1, 5), [5, 30), [30, ∞) minutes with `np.histogram`. How many fall in the last bin?

*Taught in:* session 4, block 16 (histogram and digitize).

<details><summary>Hint (open it only after a first try)</summary>

`np.histogram(g, bins=[0, 1, 5, 30, np.inf])` returns (counts, edges); the last bin is closed on the right.

</details>

In [34]:
# your code

hist, edges = np.histogram(g, bins=[0,1,5,30,np.inf])
answer_q16 = int(hist[-1])

Before you hand in: *Kernel → Restart & Run All*, and check that every answer is filled.